In [ ]:
import torch
import torch.nn as nn
import numpy as np
from src.gtsrb import NUM_CLASSES, GTSRB_CLASSES, get_dataloaders, save_predictions

In [2]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128)

In [ ]:
def train_model(optimizer_name):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),

        nn.Linear(128, NUM_CLASSES)

    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "Momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001,
            momentum=0.9
        )

    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    epochs = 5

    losses = []
    accuracies = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss

        losses.append(epoch_loss)

        # -----------------------------
        # VALIDAÇÃO
        # -----------------------------

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

        accuracy = 100 * correct / total

        accuracies.append(accuracy)

        print(
            f"{optimizer_name} | "
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Validation Accuracy: {accuracy:.2f}%"
        )

    return model, losses, accuracies

In [ ]:
def metrics(predictions, model, device):
    model.eval()

    cm = np.zeros((43,43), dtype=int)

    with torch.no_grad():
        for data in test_loader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            _, predictions = torch.max(outputs, 1)

            for label, prediction in zip(labels, predictions):
                cm[label, prediction] += 1
    
    print(cm)

    acc_class = [0 for c in GTSRB_CLASSES]
    for i in range(43):
        acc = 100 * (float(cm[i,i]) / np.sum(cm[i, :]))
        print(f'Acurácia {i}: {acc:.2f}')
        acc_class[i] = acc

    print(f"Worst class: {acc_class.index(min(acc_class))} - {min(acc_class):.2f}%")
    print(f"Best class: {acc_class.index(max(acc_class))} - {max(acc_class):.2f}%")

In [ ]:
def generate_predictions(model):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.eval()

    predictions_list = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            predictions_list.extend(
                predictions.cpu().numpy()
            )
            metrics(predictions, model, device)
    return predictions_list

In [4]:
model_sgd, losses_sgd, accs_sgd = train_model("SGD")

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SGD | Epoch 1 | Loss: 623.9647 | Validation Accuracy: 5.78%
SGD | Epoch 2 | Loss: 615.9552 | Validation Accuracy: 6.78%
SGD | Epoch 3 | Loss: 606.6674 | Validation Accuracy: 6.72%
SGD | Epoch 4 | Loss: 597.3016 | Validation Accuracy: 6.74%
SGD | Epoch 5 | Loss: 589.2455 | Validation Accuracy: 7.02%


In [5]:
model_momentum, losses_momentum, accs_momentum = train_model("Momentum")

Momentum | Epoch 1 | Loss: 600.4644 | Validation Accuracy: 10.68%
Momentum | Epoch 2 | Loss: 552.4774 | Validation Accuracy: 17.55%
Momentum | Epoch 3 | Loss: 510.0729 | Validation Accuracy: 27.42%
Momentum | Epoch 4 | Loss: 432.8326 | Validation Accuracy: 39.08%
Momentum | Epoch 5 | Loss: 342.4082 | Validation Accuracy: 51.78%


In [6]:
model_adam, losses_adam, accs_adam = train_model("Adam")

Adam | Epoch 1 | Loss: 283.8420 | Validation Accuracy: 83.11%
Adam | Epoch 2 | Loss: 56.6658 | Validation Accuracy: 93.99%
Adam | Epoch 3 | Loss: 26.6021 | Validation Accuracy: 96.81%
Adam | Epoch 4 | Loss: 14.9809 | Validation Accuracy: 98.07%
Adam | Epoch 5 | Loss: 10.8217 | Validation Accuracy: 97.05%


In [ ]:
print('Metrics for sgd')
y_pred_sgd = generate_predictions(model_sgd)

save_predictions(
    y_pred_sgd,
    "results/predicoes_sgd.csv",
    experiment_name="CNN SGD"
)

In [ ]:
print('Metrics for momentum')
y_pred_momentum = generate_predictions(model_momentum)

save_predictions(
    y_pred_momentum,
    "results/predicoes_momentum.csv",
    experiment_name="CNN Momentum"
)

In [ ]:
print('Metrics for adam')
y_pred_adam = generate_predictions(model_adam)

save_predictions(
    y_pred_adam,
    "results/predicoes_adam.csv",
    experiment_name="CNN Adam"
)